# Mars Terrain Segmentation - GPU training (Colab / Kaggle)

Trains **DeepLabV3+ (ResNet-50)** on NASA JPL **AI4Mars** (Curiosity NAVCAM, Hugging Face `hassanjbara/AI4MARS`), evaluates it on the expert-labeled test sets against an *everything is soil* baseline, and packages the checkpoint for the Mission Control app.

1. **Runtime → Change runtime type → GPU** (Colab T4 is fine; on Kaggle enable *GPU* and *Internet*).
2. Run the cells in order. Full training takes ~2-3 h on a T4; use `--max-train 6000` for a ~45 min run.
3. Download `mars_model.zip`, unzip it into the project's `models/` folder and click **Rescan models** in the dashboard.

## 1. Get the project code
Option A: clone your GitHub repo (edit the URL). Option B: upload `space-ml-mission-control.zip` with the file browser and unzip it.

In [ ]:
import os
REPO_URL = "https://github.com/<your-username>/space-ml-mission-control.git"  # <- edit, or use Option B
if not os.path.exists('space-ml-mission-control'):
    if os.path.exists('space-ml-mission-control.zip'):
        !unzip -q space-ml-mission-control.zip
    else:
        !git clone -q $REPO_URL
%cd space-ml-mission-control/backend

## 2. Install dependencies
Colab/Kaggle already ship PyTorch with CUDA.

In [ ]:
!pip install -q segmentation-models-pytorch datasets pillow scikit-learn
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## 3. Train
Labels: 0 soil, 1 bedrock, 2 sand, 3 big rock, 255 unlabeled. Rover body and >30 m range are masked out of the loss. 10% of the training images are held out for validation; the expert test sets are not touched until step 4.

In [ ]:
VERSION = 'v1-colab'
!python -m spaceml.mars.train --epochs 10 --img-size 512 --batch 8 --workers 2 --version $VERSION
# faster option: add  --max-train 6000

## 4. Evaluate on the expert-labeled test sets (min3 and min1)

In [ ]:
!python -m spaceml.mars.evaluate --version $VERSION
import json; m = json.load(open(f'../models/mars/{VERSION}/metrics.json'))
for r in m['rows']:
    print(r['group']); [print(f"  {x['name']:16s} model {x['model']:.3f}   all-soil {x['baseline']:.3f}") for x in r['metrics']]

## 5. Package for the app

In [ ]:
!cd .. && zip -qr mars_model.zip models/mars/$VERSION && ls -lh mars_model.zip
try:
    from google.colab import files; files.download('../mars_model.zip')
except ImportError:
    print('On Kaggle: download mars_model.zip from the Output panel (/kaggle/working).')
    !cp ../mars_model.zip /kaggle/working/ 2>/dev/null || true

In the app open **Mars Terrain** (or **Data & Training → Mars**) and click **Upload a trained model (.zip)**, then pick `mars_model.zip`.
The model is checked, installed as `models/mars/<version>/` and becomes active immediately - no terminal or rescan needed.